In [0]:
# Databricks Notebook: 10_Daily_Data_Generator
# 晶圆厂每日增量数据生成器（幂等：重跑结果一致）
# 用法：Databricks 界面新建 Notebook，Language 选 Python，粘贴以下全部代码
#
# v3 真实化升级：引入路线定义表（Route Definition）
#   source.raw_route_def   路线定义（product → step_seq 1..N → layer）← MES 的锚点表
#   raw_mes_moves 新增 step_seq 列，lot 沿路线推进，走完 → CP 测试
#   真实工厂：一个 route 300-800 步，一个 lot 跑 2-3 个月

from pyspark.sql import functions as F
from pyspark.sql.types import *
import random
from datetime import datetime

# =====================================================
# 配置
# =====================================================
today = datetime.now().strftime("%Y-%m-%d")
random.seed(int(datetime.now().strftime("%Y%m%d")))  # 同一天重跑结果可复现

def delete_if_exists(table, where):
    """幂等保护：首次运行时表还不存在，跳过 DELETE"""
    if spark.catalog.tableExists(table):
        spark.sql(f"DELETE FROM {table} WHERE {where}")

process_steps = ["Photo", "Etch", "CVD", "PVD", "Impl", "CMP", "Clean"]
equipment_ids = ["EQ-A01", "EQ-A02", "EQ-B01", "EQ-B02", "EQ-C01", "EQ-C02"]
equipment_types = {"EQ-A01": "Lithography", "EQ-A02": "Lithography",
                   "EQ-B01": "Etch", "EQ-B02": "Etch",
                   "EQ-C01": "CVD", "EQ-C02": "CVD"}

# =====================================================
# 路线定义（真实 MES 的 Process Plan）
#   每个产品一条 route：300+ 步，每 ~15 步为一层（layer）
#   每层必含 Photo（光刻定义层）和 Metrology（量测）→ 每层都要检测
# =====================================================
ROUTE_STEPS_PER_LAYER = 15
route_lengths = {
    "IGBT-650V": 190,       # 功率器件：13 层（厚工艺，多层外延+注入）
    "SiC-MOS-1200V": 230,   # 第三代半导体：16 层（高温离子注入+退火，工序更多）
    "BCD-PMIC": 150,        # 电源管理IC：10 层
    "SBD-Schottky": 120,    # 肖特基二极管：8 层（结构简单）
    "FRD-Fast": 120,        # 快恢复二极管：8 层
}
# 产品清单与路线定义强一致（避免 KeyError：工单产品必须存在于路线表）
products = list(route_lengths.keys())
# 一个 layer 内的典型步骤序列（15 步）
layer_pattern = ["Clean", "Photo", "Etch", "Metrology", "CVD",
                 "Impl", "CMP", "Metrology", "Clean", "PVD",
                 "CVD", "Clean", "Etch", "Metrology", "Clean"]

step_equipment = {
    "Photo": ["EQ-A01", "EQ-A02", "EQ-A03"],
    "Etch":  ["EQ-B01", "EQ-B02", "EQ-B03"],
    "CVD":   ["EQ-C01", "EQ-C02"],
    "PVD":   ["EQ-D01"],
    "Impl":  ["EQ-E01"],
    "CMP":   ["EQ-F01"],
    "Clean": ["EQ-G01", "EQ-G02"],
    "Metrology": ["EQ-H01", "EQ-H02"],
}
operators = [f"OP{str(i).zfill(3)}" for i in range(1, 11)]

print(f"📅 数据日期: {today}")
print(f"🔄 幂等模式: DELETE 当天旧数据 → APPEND 新数据")

# =====================================================
# 0. 路线定义表（静态主数据，整体覆盖）
#    面试考点：route 300+ 步、每 15 步一层、Metrology 每层出现 → SPC 数据的来源
# =====================================================
print("\n" + "=" * 50)
print("0. 路线定义表 raw_route_def")
print("=" * 50)

route_data = []
for prod, n in route_lengths.items():
    total_layers = (n + ROUTE_STEPS_PER_LAYER - 1) // ROUTE_STEPS_PER_LAYER
    for seq in range(1, n + 1):
        layer = (seq - 1) // ROUTE_STEPS_PER_LAYER + 1
        st = layer_pattern[(seq - 1) % ROUTE_STEPS_PER_LAYER]
        route_data.append((prod, seq, f"{st}_L{layer}", st, layer))

route_schema = StructType([
    StructField("product", StringType()),
    StructField("step_seq", IntegerType()),      # 全局步序 1..N
    StructField("step_name", StringType()),      # 如 Etch_L7
    StructField("step_type", StringType()),
    StructField("layer", IntegerType())          # 第几层（20-40 层）
])
route_df = spark.createDataFrame(route_data, route_schema)
route_df.write.mode("overwrite").format("delta").saveAsTable("source.raw_route_def")
print(f"✅ source.raw_route_def: {len(route_data)} 行 "
      f"({len(route_lengths)} 个产品，{min(route_lengths.values())}~{max(route_lengths.values())} 步/route)")

# =====================================================
# 1. MES 工单（每天 200 条新批次 + 10 条更新旧批次）
# =====================================================
print("\n" + "=" * 50)
print("1. MES 工单数据")
print("=" * 50)

mes_data = []
for i in range(200):
    lot_id = f"LOT{today.replace('-','')}{i:04d}"
    start_h = random.randint(0, 8)
    end_h = random.randint(start_h + 1, min(start_h + 3, 18))
    mes_data.append((
        lot_id, random.choice(equipment_ids),
        random.choice(products), random.choice(process_steps),
        25, today,
        f"{today} {start_h:02d}:{random.randint(0,59):02d}",
        f"{today} {end_h:02d}:{random.randint(0,59):02d}",
        random.choice(["completed", "completed", "completed", "hold"])
    ))

existing_holds = (spark.sql("SELECT lot_id FROM source.raw_mes WHERE status = 'hold' LIMIT 10").collect()
                  if spark.catalog.tableExists("source.raw_mes") else [])
for row in existing_holds:
    mes_data.append((
        row["lot_id"], "EQ-A01", random.choice(products), "Photo", 25,
        today, f"{today} 08:00", f"{today} 09:00", "completed"
    ))

mes_schema = StructType([
    StructField("lot_id", StringType()),
    StructField("equipment_id", StringType()),
    StructField("product", StringType()),
    StructField("process_step", StringType()),
    StructField("wafer_count", IntegerType()),
    StructField("start_date", StringType()),
    StructField("start_time", StringType()),
    StructField("end_time", StringType()),
    StructField("status", StringType())
])
mes_df = spark.createDataFrame(mes_data, mes_schema)

delete_if_exists("source.raw_mes", f"start_date = '{today}'")
mes_df.write.mode("append").format("delta").saveAsTable("source.raw_mes")
print(f"✅ source.raw_mes: +{len(mes_data)} 行，当前 {spark.table('source.raw_mes').count()} 行")

# =====================================================
# 2. 在制批次池 + 路线进度（沿 step_seq 续跑，跨天可续）
# =====================================================
print("\n" + "=" * 50)
print("2. 在制批次池与路线进度")
print("=" * 50)

# 2.1 今日新投批次：从 step_seq = 0 开始
new_lots = {item[0]: item[2] for item in mes_data[:200]}   # lot_id → product

# 2.2 在制批次：product 从工单取，进度优先取历史最大 step_seq，没有则按投片天数估算
if spark.catalog.tableExists("source.raw_mes_moves"):
    flight_rows = spark.sql(f"""
        SELECT m.lot_id, MAX(r.step_seq) AS max_seq, ANY_VALUE(m2.product) AS product
        FROM (SELECT DISTINCT lot_id FROM source.raw_mes WHERE start_date < '{today}') m
        LEFT JOIN (SELECT lot_id, product FROM source.raw_mes) m2 ON m.lot_id = m2.lot_id
        LEFT JOIN source.raw_mes_moves mv ON m.lot_id = mv.lot_id AND mv.move_date < '{today}'
        LEFT JOIN source.raw_route_def r ON mv.step_name = r.step_name
        GROUP BY m.lot_id
        LIMIT 120
    """).collect()
else:
    flight_rows = []   # 首次运行：无历史过站数据

flight_lots = {}   # lot_id → (product, start_seq)
for r in flight_rows:
    if r["max_seq"] is not None:
        flight_lots[r["lot_id"]] = (r["product"] or "IGBT-650V", int(r["max_seq"]))
    else:
        # 无过站历史：按投片天数估算进度（每天约 6 步），并回退到最近的 route 锚点
        try:
            days = (datetime.strptime(today, "%Y-%m-%d")
                    - datetime.strptime(r["lot_id"][3:11], "%Y%m%d")).days
        except Exception:
            days = 10
        est_seq = min(days * 6, 250)
        flight_lots[r["lot_id"]] = (r["product"] or "IGBT-650V", est_seq)

lot_progress = {}  # lot_id → (product, 当前 seq, 当前 qty)
for lot_id, prod in new_lots.items():
    lot_progress[lot_id] = (prod, 0, 25)
for lot_id, (prod, seq) in flight_lots.items():
    lot_progress[lot_id] = (prod, seq, 25)

print(f"   在制批次池: 今日新投 {len(new_lots)} 批 + 在制 {len(flight_lots)} 批")

# 兼容旧表结构：给 raw_mes_moves 补 step_seq 列（已存在则跳过）
try:
    spark.sql("ALTER TABLE source.raw_mes_moves ADD COLUMN step_seq INT")
    print("   已为 raw_mes_moves 扩展 step_seq 列")
except Exception:
    pass

# =====================================================
# 3. 过站明细 raw_mes_moves（沿路线推进）
#    面试考点：Track In/Track Out、re-entrant、qty 中途减少(报废)、
#              MAX(step_seq) = lot 当前进度
# =====================================================
print("\n" + "=" * 50)
print("3. MES 过站明细（move transactions）")
print("=" * 50)

moves_data = []
events_data = []
move_seq = 0
completed_lots = []   # 今天走完 route 的批次 → 触发 CP

def gen_moves_for_lot(lot_id, prod, n_moves, start_seq, qty=25):
    """lot 沿路线定义推进 n_moves 步，返回 (最新seq, qty)"""
    global move_seq
    seq = start_seq
    t_min = 8 * 60 + random.randint(0, 120)
    for _ in range(n_moves):
        if t_min >= 1440:                 # 跨天截断：剩余步骤留到明天（时间不允许 >24 小时）
            break
        seq += 1
        row = route_lookup[(prod, seq)]
        step_name, step_type = row[0], row[1]
        eq_id = random.choice(step_equipment[step_type])
        ti, to = t_min, min(t_min + random.randint(20, 90), 1439)
        qty_out = qty
        if random.random() < 0.01 and qty > 20:      # 1% 报废 1 片
            qty_out = qty - 1
            move_seq += 1
            events_data.append((
                f"EV{today.replace('-','')}{move_seq:06d}", lot_id, "SCRAP",
                f"{today} {to // 60:02d}:{to % 60:02d}:00",
                "WAFER_PROCESS_FAIL", None, 1, today
            ))
        move_seq += 1
        moves_data.append((
            f"MV{today.replace('-','')}{move_seq:06d}", lot_id,
            step_name, eq_id, step_type,
            f"RC-{step_type}-{random.randint(1,3)}",
            qty, qty_out,
            f"{today} {ti // 60:02d}:{ti % 60:02d}:00",
            f"{today} {to // 60:02d}:{to % 60:02d}:00",
            random.choice(operators), today, seq
        ))
        qty = qty_out
        t_min = to + random.randint(15, 120)         # 排队 → queue time
        if seq >= route_lengths[prod]:               # 走完全程！
            move_seq += 1
            events_data.append((
                f"EV{today.replace('-','')}{move_seq:06d}", lot_id, "ROUTE_COMPLETE",
                f"{today} {to // 60:02d}:{to % 60:02d}:00",
                "ROUTE_END", None, qty, today
            ))
            completed_lots.append((lot_id, prod))
            break
    return seq, qty

# 内存路线查找表 {(product, seq): (step_name, step_type)}
route_lookup = {(r[0], r[1]): (r[2], r[3]) for r in route_data}

for lot_id, (prod, seq, qty) in lot_progress.items():
    final_seq, final_qty = gen_moves_for_lot(lot_id, prod, random.randint(4, 8), seq, qty)
    lot_progress[lot_id] = (prod, final_seq, final_qty)

# 返工：挑 2 批重走上一道 Photo（量测超标 → 批退）
rework_lots = random.sample(list(lot_progress.keys()), min(2, len(lot_progress)))
for lot_id in rework_lots:
    prod, seq, qty = lot_progress[lot_id]
    rework_seq = max(1, seq - random.randint(3, 6))
    # 找到该序最近的 Photo 步（找不到则退到本层第一个 Photo，即 step_seq=2，防 StopIteration）
    target = next((s for s in range(rework_seq, 0, -1)
                   if route_lookup[(prod, s)][1] == "Photo"), 2)
    move_seq += 1
    events_data.append((
        f"EV{today.replace('-','')}{move_seq:06d}", lot_id, "REWORK",
        f"{today} {random.randint(13,17):02d}:{random.randint(0,59):02d}:00",
        "METROLOGY_OOC", None, qty, today
    ))
    step_name, step_type = route_lookup[(prod, target)]
    move_seq += 1
    moves_data.append((
        f"MV{today.replace('-','')}{move_seq:06d}", lot_id,
        step_name, random.choice(step_equipment[step_type]), step_type,
        f"RC-{step_type}-REWORK",
        qty, qty,
        f"{today} 16:00:00", f"{today} 17:00:00",
        random.choice(operators), today, target
    ))

# 拆批：挑 2 批拆成 10+15（谱系 parent_lot_id）
for lot_id in random.sample(list(lot_progress.keys()), min(2, len(lot_progress))):
    prod, seq, qty = lot_progress[lot_id]
    for suffix, child_qty in [("S1", 10), ("S2", qty - 10)]:
        child_id = f"{lot_id}{suffix}"
        move_seq += 1
        events_data.append((
            f"EV{today.replace('-','')}{move_seq:06d}", child_id, "SPLIT",
            f"{today} {random.randint(9,15):02d}:{random.randint(0,59):02d}:00",
            "PRIORITY_SPLIT", lot_id, child_qty, today
        ))
        lot_progress[child_id] = (prod, seq, child_qty)

# Hold/Release
for lot_id in random.sample(list(lot_progress.keys()), max(1, int(len(lot_progress) * 0.03))):
    prod, seq, qty = lot_progress[lot_id]
    move_seq += 1
    events_data.append((
        f"EV{today.replace('-','')}{move_seq:06d}", lot_id, "HOLD",
        f"{today} {random.randint(10,16):02d}:{random.randint(0,59):02d}:00",
        random.choice(["METROLOGY_OOC", "ENG_HOLD", "WIP_BALANCE"]), None, qty, today
    ))
yesterday_holds = (spark.sql(f"""
    SELECT DISTINCT lot_id FROM source.raw_lot_events
    WHERE event_type = 'HOLD' AND event_date < '{today}'
    AND lot_id NOT IN (SELECT lot_id FROM source.raw_lot_events WHERE event_type='RELEASE')
    LIMIT 5
""").collect() if spark.catalog.tableExists("source.raw_lot_events") else [])
for row in yesterday_holds:
    move_seq += 1
    events_data.append((
        f"EV{today.replace('-','')}{move_seq:06d}", row["lot_id"], "RELEASE",
        f"{today} {random.randint(9,11):02d}:{random.randint(0,59):02d}:00",
        "DISPOSITION_OK", None, 25, today
    ))

moves_schema = StructType([
    StructField("move_id", StringType()),
    StructField("lot_id", StringType()),
    StructField("step_name", StringType()),
    StructField("equipment_id", StringType()),
    StructField("step_type", StringType()),
    StructField("recipe", StringType()),
    StructField("qty_in", IntegerType()),
    StructField("qty_out", IntegerType()),
    StructField("track_in_ts", StringType()),
    StructField("track_out_ts", StringType()),
    StructField("operator_id", StringType()),
    StructField("move_date", StringType()),
    StructField("step_seq", IntegerType())           # 路线全局步序
])
moves_df = spark.createDataFrame(moves_data, moves_schema)
delete_if_exists("source.raw_mes_moves", f"move_date = '{today}'")
moves_df.write.mode("append").format("delta").saveAsTable("source.raw_mes_moves")
print(f"✅ source.raw_mes_moves: +{len(moves_data)} 行（{len(lot_progress)} 个 lot 沿路线推进）")

# =====================================================
# 4. 批次事件表 raw_lot_events
# =====================================================
events_schema = StructType([
    StructField("event_id", StringType()),
    StructField("lot_id", StringType()),
    StructField("event_type", StringType()),         # HOLD/RELEASE/SPLIT/REWORK/SCRAP/ROUTE_COMPLETE
    StructField("event_ts", StringType()),
    StructField("reason_code", StringType()),
    StructField("parent_lot_id", StringType()),
    StructField("qty", IntegerType()),
    StructField("event_date", StringType())
])
events_df = spark.createDataFrame(events_data, events_schema)
delete_if_exists("source.raw_lot_events", f"event_date = '{today}'")
events_df.write.mode("append").format("delta").saveAsTable("source.raw_lot_events")
type_counts = {}
for e in events_data:
    type_counts[e[2]] = type_counts.get(e[2], 0) + 1
print(f"✅ source.raw_lot_events: +{len(events_data)} 行 {type_counts}")

# =====================================================
# 5. CP 测试 bin 分档（只测今天走完 route 的批次！）
#    面试考点：CP 发生在 route 结束后；良率 = BIN1 占比
# =====================================================
print("\n" + "=" * 50)
print("4. CP 测试 bin 数据")
print("=" * 50)

cp_data = []
cp_lots = completed_lots if completed_lots else \
    [(lot, prod) for lot, (prod, seq, _) in list(lot_progress.items())
     if seq >= route_lengths[prod] - 3][:3]          # 前几天跑完的也补测
for lot_id, prod in cp_lots:
    for w in range(1, 26):
        wafer_id = f"{lot_id}W{w:02d}"
        dies = 300
        cp_yield = random.uniform(0.82, 0.98)
        bin1 = int(dies * cp_yield)
        rest = dies - bin1
        bins = [("BIN1", "PASS", bin1)]
        for i, b in enumerate(range(2, 9)):
            if rest <= 0:
                break
            cnt = rest if i == 5 else random.randint(0, max(1, rest // 2))
            cnt = min(cnt, rest)
            rest -= cnt
            if cnt > 0:
                bins.append((f"BIN{b}", f"FAIL_CAT{b}", cnt))
        for bin_code, bin_desc, cnt in bins:
            cp_data.append((
                wafer_id, lot_id, prod, bin_code, bin_desc, cnt,
                round(bin1 / dies * 100, 2), today
            ))

cp_schema = StructType([
    StructField("wafer_id", StringType()),
    StructField("lot_id", StringType()),
    StructField("product", StringType()),
    StructField("bin_code", StringType()),
    StructField("bin_desc", StringType()),
    StructField("die_count", IntegerType()),
    StructField("cp_yield_pct", DoubleType()),
    StructField("test_date", StringType())
])
cp_df = spark.createDataFrame(cp_data, cp_schema)
delete_if_exists("source.raw_cp_bins", f"test_date = '{today}'")
cp_df.write.mode("append").format("delta").saveAsTable("source.raw_cp_bins")
print(f"✅ source.raw_cp_bins: +{len(cp_data)} 行（CP 完工批: {len(cp_lots)} 个）")

# =====================================================
# 6. 设备状态流转 raw_equip_state（OEE 数据源）
# =====================================================
print("\n" + "=" * 50)
print("5. 设备状态流转")
print("=" * 50)

all_equip = sorted({eq for eqs in step_equipment.values() for eq in eqs})
state_data = []
for eq_id in all_equip:
    t_min = 0
    while t_min < 1439:   # 1439 及以上视为已到 23:59，避免 end 封顶后 t_min 无法前进导致死循环
        r = random.random()
        if r < 0.60:
            state, dur, reason = "RUN", random.randint(30, 120), None
        elif r < 0.75:
            state, dur, reason = "IDLE", random.randint(10, 60), None
        elif r < 0.85:
            state, dur = "DOWN", random.randint(30, 240)
            reason = random.choice(["BREAKDOWN", "ALARM"])
        else:
            state, dur, reason = "PM", random.randint(60, 180), "SCHEDULED_PM"
        end = min(t_min + dur, 1439)   # 封顶 23:59，避免拼出 "24:00" 非法时间
        state_data.append((
            eq_id, state,
            f"{today} {t_min // 60:02d}:{t_min % 60:02d}:00",
            f"{today} {end // 60:02d}:{end % 60:02d}:00",
            end - t_min, reason, today
        ))
        t_min = end

state_schema = StructType([
    StructField("equipment_id", StringType()),
    StructField("state", StringType()),
    StructField("start_ts", StringType()),
    StructField("end_ts", StringType()),
    StructField("duration_min", IntegerType()),
    StructField("reason", StringType()),
    StructField("state_date", StringType())
])
state_df = spark.createDataFrame(state_data, state_schema)
delete_if_exists("source.raw_equip_state", f"state_date = '{today}'")
state_df.write.mode("append").format("delta").saveAsTable("source.raw_equip_state")
print(f"✅ source.raw_equip_state: +{len(state_data)} 行（{len(all_equip)} 台设备）")

# =====================================================
# 7. 质检数据
# =====================================================
print("\n" + "=" * 50)
print("6. 质检数据")
print("=" * 50)

quality_data = []
for item in mes_data[:200]:
    lot_id = item[0]
    total_dies = random.randint(100, 500)   # 6英寸功率器件：die 大数量少
    yield_rate = random.uniform(0.85, 0.99) if random.random() > 0.03 else random.uniform(0.5, 0.7)
    defect_type = random.choice(["Particle", "Scratch", "Pattern Defect", "Oxide Breakdown", "Contamination"]) if yield_rate < 0.95 else None
    quality_data.append((
        f"QC{lot_id}", lot_id,
        round(yield_rate * 100, 2), total_dies,
        int(total_dies * (1 - yield_rate)),
        defect_type, today
    ))

quality_schema = StructType([
    StructField("qc_id", StringType()),
    StructField("lot_id", StringType()),
    StructField("yield_rate", DoubleType()),
    StructField("total_dies", IntegerType()),
    StructField("defect_count", IntegerType()),
    StructField("primary_defect", StringType()),
    StructField("inspection_date", StringType())
])
quality_df = spark.createDataFrame(quality_data, quality_schema)

delete_if_exists("source.raw_quality", f"inspection_date = '{today}'")
quality_df.write.mode("append").format("delta").saveAsTable("source.raw_quality")
print(f"✅ source.raw_quality: +{len(quality_data)} 行，当前 {spark.table('source.raw_quality').count()} 行")

# =====================================================
# 8. 设备传感器（每小时快照）
# =====================================================
print("\n" + "=" * 50)
print("7. 设备传感器数据")
print("=" * 50)

sensor_data = []
for eq_id in equipment_ids:
    for hour in range(24):
        temp = random.gauss(25, 2)
        pressure = random.gauss(1013, 10)
        rf_power = random.gauss(500, 50)
        if random.random() < 0.02:
            temp += random.choice([-50, 50])
            pressure += random.choice([-200, 200])
        sensor_data.append((
            f"TS{today.replace('-','')}{eq_id}{hour:02d}",
            eq_id, equipment_types[eq_id],
            f"{today} {hour:02d}:00:00",
            round(temp, 2), round(pressure, 2), round(rf_power, 2),
            random.choice(["running", "running", "idle", "maintenance"])
        ))

sensor_schema = StructType([
    StructField("sensor_id", StringType()),
    StructField("equipment_id", StringType()),
    StructField("equipment_type", StringType()),
    StructField("timestamp", StringType()),
    StructField("temperature", DoubleType()),
    StructField("pressure", DoubleType()),
    StructField("rf_power", DoubleType()),
    StructField("status", StringType())
])
sensor_df = spark.createDataFrame(sensor_data, sensor_schema)

delete_if_exists("source.raw_sensors", f"SUBSTRING(timestamp, 1, 10) = '{today}'")
sensor_df.write.mode("append").format("delta").saveAsTable("source.raw_sensors")
print(f"✅ source.raw_sensors: +{len(sensor_data)} 行，当前 {spark.table('source.raw_sensors').count()} 行")

# =====================================================
# 9. 验证幂等性
# =====================================================
print("\n" + "=" * 50)
print("📊 source 层当前总行数")
print("=" * 50)
for table in ["raw_route_def", "raw_mes", "raw_mes_moves", "raw_lot_events",
              "raw_cp_bins", "raw_equip_state", "raw_quality", "raw_sensors", "raw_products"]:
    cnt = spark.table(f"source.{table}").count()
    print(f"  source.{table}: {cnt}")

print(f"\n🎉 {today} 造数完成！重跑本 Notebook 结果完全一致（幂等）")
print("👉 下一步：运行 Bronze → Silver → Gold 管道")
print("\n💡 验证真实 route 的三个自测查询：")
print("   ① 路线长度: SELECT product, MAX(step_seq) AS steps FROM source.raw_route_def GROUP BY 1 ORDER BY 2 DESC")
print("   ② lot 进度: SELECT lot_id, MAX(step_seq) AS progress FROM source.raw_mes_moves GROUP BY 1 ORDER BY 2 DESC LIMIT 10")
print("   ③ 重入流: SELECT step_type, COUNT(DISTINCT layer) AS layers FROM source.raw_route_def GROUP BY 1")


📅 数据日期: 2026-09-03
🔄 幂等模式: DELETE 当天旧数据 → APPEND 新数据

0. 路线定义表 raw_route_def
✅ source.raw_route_def: 810 行 (5 个产品，120~230 步/route)

1. MES 工单数据
✅ source.raw_mes: +210 行，当前 210 行

2. 在制批次池与路线进度
   在制批次池: 今日新投 200 批 + 在制 0 批


{"ts": "2026-09-03 09:54:29.333", "level": "ERROR", "logger": "SQLQueryContextLogger", "msg": "[FIELD_ALREADY_EXISTS] Cannot add column, because `step_seq` already exists in \"STRUCT<move_id: STRING, lot_id: STRING, step_name: STRING, equipment_id: STRING, step_type: STRING, recipe: STRING, qty_in: INT, qty_out: INT, track_in_ts: STRING, track_out_ts: STRING, operator_id: STRING, move_date: STRING, step_seq: INT>\". SQLSTATE: 42710; line 1 pos 0;\nAddColumns [QualifiedColType(None,step_seq,IntegerType,true,None,None,None,None)]\n+- ResolvedTable com.databricks.sql.managedcatalog.UnityCatalogV2Proxy@672811d4, source.raw_mes_moves, DeltaTableV2(org.apache.spark.sql.SparkSession@6bdadc84,abfss://unity-catalog-storage@dbstoragew4jpdufwyixte.dfs.core.windows.net/7405609498150470/__unitystorage/catalogs/1c0b37ec-c3a8-4bd5-b7d4-69524e9d37d3/tables/aad3e7ae-e59b-4318-abfc-07de06436d27,Some(CatalogTable(\nCatalog: adb_fab_etl\nDatabase: source\nTable: raw_mes_moves\nOwner: 13535507592@139.com\n


3. MES 过站明细（move transactions）
✅ source.raw_mes_moves: +1216 行（204 个 lot 沿路线推进）
✅ source.raw_lot_events: +21 行 {'SCRAP': 9, 'REWORK': 2, 'SPLIT': 4, 'HOLD': 6}

4. CP 测试 bin 数据
✅ source.raw_cp_bins: +0 行（CP 完工批: 0 个）

5. 设备状态流转
✅ source.raw_equip_state: +282 行（15 台设备）

6. 质检数据
✅ source.raw_quality: +200 行，当前 200 行

7. 设备传感器数据
✅ source.raw_sensors: +144 行，当前 144 行

📊 source 层当前总行数
  source.raw_route_def: 810
  source.raw_mes: 210
  source.raw_mes_moves: 1216
  source.raw_lot_events: 21
  source.raw_cp_bins: 0
  source.raw_equip_state: 282
  source.raw_quality: 200
  source.raw_sensors: 144
  source.raw_products: 5

🎉 2026-09-03 造数完成！重跑本 Notebook 结果完全一致（幂等）
👉 下一步：运行 Bronze → Silver → Gold 管道

💡 验证真实 route 的三个自测查询：
   ① 路线长度: SELECT product, MAX(step_seq) AS steps FROM source.raw_route_def GROUP BY 1 ORDER BY 2 DESC
   ② lot 进度: SELECT lot_id, MAX(step_seq) AS progress FROM source.raw_mes_moves GROUP BY 1 ORDER BY 2 DESC LIMIT 10
   ③ 重入流: SELECT step_type, COUNT(DISTINCT layer) AS layer